# Entity Extraction [Step 2 - Turning Prose Into Triples]

> **MLCourse - Agentic AI - Advanced RAG - Graph RAG**

A knowledge graph is only as good as its extraction step. This notebook builds
that step: an LLM reads a passage and emits structured
`(subject, relation, object)` triples.

Three things make the difference between a usable graph and a useless one, and
we handle each explicitly:

1. **A constrained schema** - without one, the model invents a new relation name
   for every sentence and nothing ever connects.
2. **Reliable parsing** - the output must survive being turned into data.
3. **Entity resolution** - "the Queen", "Queen of Hearts" and "Her Majesty" must
   collapse into one node.

### 1. Setup


In [1]:
import os
import re
import time
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
from dotenv import load_dotenv


def find_env(start=None):
    """Walk up from the notebook directory until a .env file appears."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        candidate = folder / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("No .env found walking up from " + str(start))


ENV_PATH = find_env()
load_dotenv(ENV_PATH)
DATA_DIR = ENV_PATH.parent / "data"

print("env file :", ENV_PATH)
print("data dir :", DATA_DIR)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))

env file : D:\projects\python\MLCourse\03_agentic_ai\.env
data dir : D:\projects\python\MLCourse\03_agentic_ai\data
GROQ_API_KEY present: True


In [2]:
from langchain_groq import ChatGroq

GROQ_MODEL = "qwen/qwen3.8-27b"          # verified available on this account
llm = ChatGroq(model=GROQ_MODEL, temperature=0)

THINK_RE = re.compile(r"<think>.*?</think>", re.DOTALL)


def clean(text):
    """Strip any <think>...</think> block a reasoning model may emit."""
    return THINK_RE.sub("", text).strip()


def ask(prompt, retries=4, pause=1.5):
    """Call Groq with exponential backoff. Free tier is roughly 8000 tokens/minute,
    so every loop in these notebooks paces itself and retries on rate limits."""
    delay = 5.0
    for attempt in range(retries):
        try:
            answer = clean(llm.invoke(prompt).content)
            time.sleep(pause)
            return answer
        except Exception as exc:
            if attempt == retries - 1:
                raise
            print(f"  [retry {attempt + 1}] {type(exc).__name__} - sleeping {delay:.0f}s")
            time.sleep(delay)
            delay *= 2


print("Groq model:", GROQ_MODEL)
print("smoke test:", ask("Reply with exactly one word: ready"))

Groq model: qwen/qwen3.8-27b


smoke test: ready


In [3]:
ALICE_PATH = DATA_DIR / "alice.txt"
raw_text = ALICE_PATH.read_text(encoding="utf-8-sig")

paragraphs = [" ".join(p.split()) for p in raw_text.split("\n\n") if len(p.strip()) > 200]
print("paragraphs:", len(paragraphs))

paragraphs: 237


### 2. Why the prompt needs a schema

Compare two extraction prompts on the same passage. The first is unconstrained;
the second fixes the entity types and the relation vocabulary.

The unconstrained version produces triples that look fine individually and refuse
to join up into a graph, because `owns` / `has` / `is the owner of` are three
distinct edges to a machine.

In [4]:
SAMPLE = next(p for p in paragraphs if "Cheshire" in p and len(p) > 400)
print(SAMPLE[:420], "...")

So she set the little creature down, and felt quite relieved to see it trot away quietly into the wood. “If it had grown up,” she said to herself, “it would have made a dreadfully ugly child: but it makes rather a handsome pig, I think.” And she began thinking over other children she knew, who might do very well as pigs, and was just saying to herself, “if one only knew the right way to change them—” when she was a l ...


In [5]:
LOOSE_PROMPT = (
    "Extract (subject, relation, object) triples from this passage. "
    "One per line as subject | relation | object.\n\n{text}"
)

loose = ask(LOOSE_PROMPT.format(text=SAMPLE))
print("UNCONSTRAINED EXTRACTION:\n")
print(loose)

UNCONSTRAINED EXTRACTION:

she | set | the little creature
the little creature | trotted away | into the wood
she | felt | relieved
it | would have made | a dreadfully ugly child
it | makes | a handsome pig
she | began thinking over | other children
she | was saying | if one only knew the right way to change them
she | was startled by | seeing the Cheshire Cat
the Cheshire Cat | was sitting on | a bough of a tree
the bough | was on | a tree
the tree | was | a few yards off


### The constrained version

Four rules do most of the work:

- **Fix the entity types** the model may use, so it cannot invent categories.
- **Fix the relation vocabulary**, or at least demand short lowercase verb
  phrases from a listed set.
- **Demand canonical names** - full names, no pronouns - which is entity
  resolution done at extraction time rather than as cleanup afterwards.
- **Demand a strict output format**, so parsing does not become guesswork.

In [6]:
ENTITY_TYPES = ["Character", "Place", "Object", "Event"]
RELATIONS = ["meets", "attends", "owns", "belongs to", "located in", "causes",
             "orders", "accuses", "testifies at", "gives", "eats", "drinks",
             "follows", "carries", "plays", "presides over"]

STRICT_PROMPT = (
    "You extract a knowledge graph from Lewis Carroll's 'Alice's Adventures in "
    "Wonderland'.\n\n"
    "Entity types you may use: " + ", ".join(ENTITY_TYPES) + "\n"
    "Relations you may use: " + ", ".join(RELATIONS) + "\n\n"
    "Rules:\n"
    "- Use ONLY the relations listed above. If nothing fits, skip the fact.\n"
    "- Use canonical full names: 'Queen of Hearts' not 'the Queen', 'Alice' not "
    "'she'. Never output a pronoun as an entity.\n"
    "- One triple per line, in exactly this format:\n"
    "  subject | relation | object\n"
    "- No numbering, no commentary, no blank lines.\n\n"
    "Passage:\n{text}\n\nTriples:"
)

strict = ask(STRICT_PROMPT.format(text=SAMPLE))
print("CONSTRAINED EXTRACTION:\n")
print(strict)

CONSTRAINED EXTRACTION:

Alice | carries | Cheshire Cat
Cheshire Cat | located in | Wood
Cheshire Cat | located in | Tree


### 3. Parsing defensively

Never assume the model obeyed the format. Real extraction pipelines drop
malformed lines silently and count them - a rising malformed rate is your signal
that a prompt or a model change has broken something.

In [7]:
def parse_triples(text, allowed_relations=None):
    """Parse 'subject | relation | object' lines, dropping anything malformed."""
    good, bad = [], []
    for line in text.splitlines():
        line = line.strip().lstrip("-*0123456789. ")
        if not line:
            continue
        parts = [p.strip() for p in line.split("|")]
        if len(parts) != 3 or not all(parts):
            bad.append(line)
            continue
        subject, relation, obj = parts
        relation = relation.lower()
        if allowed_relations and relation not in allowed_relations:
            bad.append(line + "   <- relation not in schema")
            continue
        good.append((subject, relation, obj))
    return good, bad


parsed, rejected = parse_triples(strict, allowed_relations=set(RELATIONS))

print(f"accepted {len(parsed)} triples, rejected {len(rejected)} lines\n")
for s, r, o in parsed:
    print(f"  ({s}) --[{r}]--> ({o})")
if rejected:
    print("\nrejected:")
    for line in rejected:
        print("  ", line[:90])

accepted 3 triples, rejected 0 lines

  (Alice) --[carries]--> (Cheshire Cat)
  (Cheshire Cat) --[located in]--> (Wood)
  (Cheshire Cat) --[located in]--> (Tree)


### 4. Extracting over a batch of passages

Extraction is a **one-off indexing cost**: you pay it once per document, then
query the graph forever. That framing is what makes an LLM call per chunk
acceptable.

We process a small sample here and pace the loop - the Groq free tier is roughly
8000 tokens per minute, and extraction prompts carry a full passage each.

In [8]:
import random

random.seed(7)
SAMPLE_IDS = sorted(random.sample(range(len(paragraphs)), 8))

all_triples, malformed = [], 0
for n, doc_id in enumerate(SAMPLE_IDS, 1):
    out = ask(STRICT_PROMPT.format(text=paragraphs[doc_id][:1400]))
    good, bad = parse_triples(out, allowed_relations=set(RELATIONS))
    malformed += len(bad)
    for triple in good:
        all_triples.append((*triple, doc_id))     # keep provenance!
    print(f"[{n}/{len(SAMPLE_IDS)}] doc_{doc_id}: {len(good)} triples, {len(bad)} rejected")
    time.sleep(3.0)          # pacing for the 8000 tokens/minute free tier

print(f"\ntotal: {len(all_triples)} triples, {malformed} malformed lines")

[1/8] doc_12: 7 triples, 0 rejected


[2/8] doc_18: 3 triples, 0 rejected


[3/8] doc_38: 3 triples, 0 rejected


[4/8] doc_82: 1 triples, 0 rejected


[5/8] doc_101: 1 triples, 0 rejected


[6/8] doc_137: 2 triples, 0 rejected


[7/8] doc_166: 1 triples, 0 rejected


[8/8] doc_210: 2 triples, 3 rejected



total: 20 triples, 3 malformed lines


Note the fourth element in each record: **`doc_id`**. Always store which chunk a
triple came from. It gives you three things you will need:

- **Citations** - the final answer can point at the source text.
- **Debuggability** - when a traversal returns nonsense you can read the sentence
  that produced the bad edge.
- **Incremental updates** - when a document changes, delete its triples by
  `doc_id` and re-extract just that document.

### 5. Entity resolution

Even with canonical-name instructions, aliases leak through. Two cheap
techniques catch most of it: a **normalisation pass** (case, articles,
whitespace) and an **alias map** for the domain's known synonyms.

The expensive technique - embedding every entity name and clustering by cosine
similarity - is worth it at scale, but start with the cheap ones.

In [9]:
ALIASES = {
    "the queen": "Queen of Hearts", "queen": "Queen of Hearts",
    "her majesty": "Queen of Hearts",
    "the king": "King of Hearts", "king": "King of Hearts",
    "the rabbit": "White Rabbit", "rabbit": "White Rabbit",
    "the cat": "Cheshire Cat", "cat": "Cheshire Cat",
    "the hatter": "Mad Hatter", "hatter": "Mad Hatter",
    "the duchess": "Duchess", "the caterpillar": "Caterpillar",
    "the dormouse": "Dormouse", "the march hare": "March Hare",
    "the gryphon": "Gryphon", "the mock turtle": "Mock Turtle",
    "the knave": "Knave of Hearts", "the knave of hearts": "Knave of Hearts",
}


def canonical(name):
    key = " ".join(name.strip().lower().split())
    key = re.sub(r"^(the|a|an)\s+", "", key)
    if key in ALIASES:
        return ALIASES[key]
    if ("the " + key) in ALIASES:
        return ALIASES["the " + key]
    return name.strip()


raw_names = sorted({t[0] for t in all_triples} | {t[2] for t in all_triples})
resolved = sorted({canonical(n) for n in raw_names})

print(f"distinct entity strings before resolution: {len(raw_names)}")
print(f"distinct entities after resolution        : {len(resolved)}\n")
for name in raw_names:
    c = canonical(name)
    if c != name:
        print(f"  '{name}'  ->  '{c}'")

distinct entity strings before resolution: 19
distinct entities after resolution        : 19



In [10]:
clean_triples = [(canonical(s), r.lower(), canonical(o), doc_id)
                 for s, r, o, doc_id in all_triples]

# Deduplicate: the same fact stated in two passages is one edge with two sources.
merged = {}
for s, r, o, doc_id in clean_triples:
    merged.setdefault((s, r, o), set()).add(doc_id)

print(f"{len(clean_triples)} raw triples -> {len(merged)} distinct edges\n")
for (s, r, o), sources in list(merged.items())[:12]:
    print(f"  ({s}) --[{r}]--> ({o})   sources: {sorted(sources)}")

20 raw triples -> 20 distinct edges

  (Alice) --[meets]--> (three-legged table)   sources: [12]
  (tiny golden key) --[located in]--> (three-legged table)   sources: [12]
  (Alice) --[owns]--> (tiny golden key)   sources: [12]
  (tiny golden key) --[belongs to]--> (doors of the hall)   sources: [12]
  (Alice) --[meets]--> (low curtain)   sources: [12]
  (Alice) --[meets]--> (little door)   sources: [12]
  (Alice) --[carries]--> (tiny golden key)   sources: [12]
  (Alice) --[carries]--> (little golden key)   sources: [18]
  (little golden key) --[located in]--> (table)   sources: [18]
  (Alice) --[located in]--> (garden)   sources: [18]
  (Alice) --[meets]--> (Mouse)   sources: [38]
  (Alice) --[owns]--> (terrier)   sources: [38]


### 6. Quality check: does the graph say what the text says?

Extraction can hallucinate. A cheap, effective audit is to sample a few
extracted triples and ask the model to verify each against its own source
paragraph - an instance of the grading idea from
[`../05_corrective_rag`](../05_corrective_rag/README.md).

In [11]:
checks = list(merged.items())[:4]
report = []
for (s, r, o), sources in checks:
    doc_id = sorted(sources)[0]
    verdict = ask(
        "Does the passage below support this claim? Answer with exactly one word, "
        "SUPPORTED or NOT_SUPPORTED, then a one-sentence reason.\n\n"
        f"Claim: {s} {r} {o}\n\nPassage: {paragraphs[doc_id][:900]}"
    )
    report.append(((s, r, o), doc_id, verdict))
    time.sleep(2.5)

for triple, doc_id, verdict in report:
    print(f"({triple[0]}) --[{triple[1]}]--> ({triple[2]})   [doc_{doc_id}]")
    print("   ->", verdict.replace("\n", " ")[:180])
    print()

(Alice) --[meets]--> (three-legged table)   [doc_12]
   -> SUPPORTED The passage explicitly states that Alice "came upon a little three-legged table," which directly confirms the claim that she meets one.

(tiny golden key) --[located in]--> (three-legged table)   [doc_12]
   -> SUPPORTED The passage explicitly states that on the little three-legged table, there was nothing except a tiny golden key.

(Alice) --[owns]--> (tiny golden key)   [doc_12]
   -> NOT_SUPPORTED The passage describes Alice finding and using the key, but does not state that she owns it.

(tiny golden key) --[belongs to]--> (doors of the hall)   [doc_12]
   -> NOT_SUPPORTED, the passage states that the key did not open any of the doors of the hall, only a separate little door behind a curtain.



### 7. Pitfalls

- **Unconstrained relations kill the graph.** Without a fixed vocabulary you get
  a graph where nothing connects because every edge is uniquely named. This is
  the single most common Graph RAG failure.
- **Pronoun subjects.** `("she", "meets", "Cheshire Cat")` is worthless.
  Instruct against it and reject it at parse time.
- **Always keep provenance.** No `doc_id`, no citations and no way to debug.
- **Extraction is lossy by design.** Anything conditional, temporal or hedged
  does not survive the trip into a triple. Keep the original text - the hybrid
  system in notebook 05 depends on having both.
- **Budget the indexing pass.** One LLM call per chunk over 10,000 chunks is a
  real cost; batch it offline, cache by content hash, and re-extract only changed
  documents.

### 8. Key takeaways

- Constrain the schema: fixed entity types, fixed relation vocabulary, canonical
  names, strict output format.
- Parse defensively and count what you reject - the malformed rate is a health
  metric.
- Store `doc_id` provenance with every triple.
- Resolve entities with normalisation plus an alias map before you build the
  graph, not after.

Next: [`03_building_the_graph.ipynb`](03_building_the_graph.ipynb) - assembling
triples into a NetworkX graph and inspecting its structure.